In [1]:
# 코드 활용해서 구글 마운트 하기(내 드라이브 접근하기)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 현재 작업 디렉토리 환경 확인하기
%pwd

'/content'

In [3]:
# 작업 디렉토리 변경하기
%cd /content/drive/MyDrive/핵심프로젝트

/content/drive/MyDrive/핵심프로젝트


In [4]:
# 지금 위치와 data 폴더 안 확인
!pwd
!ls data

/content/drive/MyDrive/핵심프로젝트
cataract_cropped.zip  normal_cropped.zip


In [2]:
!unzip -q "/content/drive/MyDrive/핵심프로젝트/data/cataract_cropped.zip" -d /content/dataset
!unzip -q "/content/drive/MyDrive/핵심프로젝트/data/normal_cropped.zip" -d /content/dataset
!ls /content/dataset

cataract  cataract_cropped  normal  normal_cropped


In [3]:
!find /content/drive/MyDrive/핵심프로젝트 -name "*.zip"

/content/drive/MyDrive/핵심프로젝트/data/cataract_cropped.zip
/content/drive/MyDrive/핵심프로젝트/data/normal_cropped.zip


In [4]:
!unzip -l "/content/drive/MyDrive/핵심프로젝트/data/cataract_cropped.zip" | head -10
!unzip -l "/content/drive/MyDrive/핵심프로젝트/data/normal_cropped.zip" | head -10

Archive:  /content/drive/MyDrive/핵심프로젝트/data/cataract_cropped.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
        0  2026-09-15 02:28   cataract_cropped/
        0  2026-09-15 02:22   cataract_cropped/cataract/
    39112  2026-09-15 02:21   cataract_cropped/cataract/10_JPG_jpg.rf.8bd4bf1a5b100a66050e5b1a1747a5ef.jpg
    31665  2026-09-15 02:21   cataract_cropped/cataract/16_jpg.rf.5c781b1c4b0fffeed3ce2be47f54fed1.jpg
    34115  2026-09-15 02:21   cataract_cropped/cataract/16_jpg.rf.e96a758429e84308092d4c0739f2960f.jpg
    60319  2026-09-15 02:21   cataract_cropped/cataract/17_jpg.rf.4aeca1f1970530d325a32e02f895b8a9.jpg
    24811  2026-09-15 02:21   cataract_cropped/cataract/18_jpg.rf.1fe5839997eb4880a2a2c340dd3df50f.jpg
Archive:  /content/drive/MyDrive/핵심프로젝트/data/normal_cropped.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
        0  2026-09-15 02:29   normal_cropped/
        0  2026-09-15 02:30   normal_cropped/normal/
     44

In [5]:
# 이전 폴더 정리
!rm -rf /content/dataset /content/tmp_unzip

# 임시 폴더에 압축 풀기
!unzip -q "/content/drive/MyDrive/핵심프로젝트/data/cataract_cropped.zip" -d /content/tmp_unzip
!unzip -q "/content/drive/MyDrive/핵심프로젝트/data/normal_cropped.zip" -d /content/tmp_unzip

# 안쪽 클래스 폴더만 dataset으로 옮기기
!mkdir -p /content/dataset
!mv /content/tmp_unzip/cataract_cropped/cataract /content/dataset/
!mv /content/tmp_unzip/normal_cropped/normal /content/dataset/

# 임시 폴더 삭제 후 확인
!rm -rf /content/tmp_unzip
!ls /content/dataset

cataract  normal


In [7]:
# 데이터 개수 확인하기
!ls /content/dataset/cataract | wc -l
!ls /content/dataset/normal | wc -l

792
765


normal 쪽에 04b03...jpg와 04b03... (1).jpg, 10.jpg와 10 - Copy.jpg처럼 복사본으로 보이는 파일이 있습니다. cataract 쪽의 16_jpg.rf.5c78...와 16_jpg.rf.e96a...는 Roboflow에서 같은 원본(16번)으로 만든 변형본일 가능성이 높아요.

이런 사진이 train/val 분할 과정에서 양쪽으로 나뉘어 들어가면, 모델이 검증 때 사실상 이미 본 사진을 맞히는 것이 되어 정확도가 실제보다 높게 나옵니다. 발표나 보고서에서 성능 수치를 쓸 때 문제가 될 수 있어요.

In [8]:
!pip -q install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 8.6 MB/s eta 0:00:00


In [9]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [10]:
from ultralytics import YOLO
model = YOLO("yolo11n-cls.pt")
print(model.task)

classify


In [11]:
!ls /content/dataset_split
!ls /content/dataset_split/train /content/dataset_split/val

ls: cannot access '/content/dataset_split': No such file or directory
ls: cannot access '/content/dataset_split/train': No such file or directory
ls: cannot access '/content/dataset_split/val': No such file or directory


In [12]:
!ls /content/dataset

cataract  normal


In [13]:
import random, re, shutil, subprocess
from pathlib import Path
from collections import defaultdict
from google.colab import drive

# 1) 드라이브 연결
drive.mount('/content/drive')

ZIP_DIR = "/content/drive/MyDrive/핵심프로젝트/data"
SRC = Path("/content/dataset")
DST = Path("/content/dataset_split")
TMP = Path("/content/tmp_unzip")
VAL_RATIO = 0.2   # 팀에서 정한 비율로 맞추세요
random.seed(42)

# 2) 압축 풀기 (dataset이 없을 때만)
if not SRC.exists():
    shutil.rmtree(TMP, ignore_errors=True)
    for name in ["cataract", "normal"]:
        subprocess.run(["unzip", "-q", f"{ZIP_DIR}/{name}_cropped.zip", "-d", str(TMP)], check=True)
        SRC.mkdir(parents=True, exist_ok=True)
        shutil.move(str(TMP / f"{name}_cropped" / name), str(SRC / name))
    shutil.rmtree(TMP, ignore_errors=True)
    print("압축 풀기 완료")
else:
    print("dataset 이미 있음 - 압축 풀기 건너뜀")

# 3) train/val 분할 (같은 원본에서 나온 사진은 한쪽에만)
def group_key(filename):
    stem = Path(filename).stem
    stem = stem.split(".rf.")[0]
    stem = re.sub(r"( - Copy| \(\d+\))+$", "", stem)
    return stem

IMG_EXT = {".jpg", ".jpeg", ".png"}
shutil.rmtree(DST, ignore_errors=True)

for cls_dir in sorted(SRC.iterdir()):
    groups = defaultdict(list)
    for f in cls_dir.iterdir():
        if f.suffix.lower() in IMG_EXT:
            groups[group_key(f.name)].append(f)

    keys = sorted(groups)
    random.shuffle(keys)
    total = sum(len(v) for v in groups.values())

    val_count = 0
    for k in keys:
        split = "val" if val_count < total * VAL_RATIO else "train"
        if split == "val":
            val_count += len(groups[k])
        out = DST / split / cls_dir.name
        out.mkdir(parents=True, exist_ok=True)
        for f in groups[k]:
            shutil.copy(f, out / f.name)

# 4) 결과 확인
for split in ["train", "val"]:
    for cls in ["cataract", "normal"]:
        n = sum(1 for f in (DST / split / cls).iterdir() if f.suffix.lower() in IMG_EXT)
        print(f"{split}/{cls}: {n}장")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
dataset 이미 있음 - 압축 풀기 건너뜀
train/cataract: 633장
train/normal: 612장
val/cataract: 159장
val/normal: 153장


In [14]:
results = model.train(
    data="/content/dataset_split",
    epochs=50,
    imgsz=224,          # 팀에서 정한 입력 크기가 있으면 그 값으로
    batch=32,
    patience=10,
    seed=42,
    device=0,
    project="/content/drive/MyDrive/핵심프로젝트/runs",
    name="cataract_cls",
)

Ultralytics 8.4.152 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_split, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=cataract_cls, nbs=64, nms=None, opset=None, optim